# 03. RAG 색인 구축 (Chroma + BM25)

사업보고서 본문을 청킹하여 벡터 DB(Chroma)와 BM25 색인에 동시 저장합니다.


## 3.1 환경 설정

In [ ]:
import os, sys
from google.colab import userdata
sys.path.insert(0, '/content/repo')

# OpenAI API 키 (임베딩용)
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("OpenAI API 키 로드")
except:
    print("⚠️ Colab Secrets에 OPENAI_API_KEY 등록 필요")
    raise

# 경로
PROCESSED_DIR = '/content/data/processed'
DB_DIR = '/content/db'

## 3.2 RAG 색인 구축

In [ ]:
from src.data.build_rag_db import build_rag_index

stats = build_rag_index(
    sections_path=f'{PROCESSED_DIR}/sections.jsonl',
    output_dir=f'{DB_DIR}/chroma_rag',
    embedding_model='text-embedding-3-small',
    chunk_size=800,      # 평균 800자
    chunk_overlap=100,   # 100자 겹침
)

print(f"\nRAG 색인 완료")
print(f"  청크 수: {stats['chunks']:,}개")
print(f"  색인 크기: {stats['size_mb']:.1f} MB")

## 3.3 검색 테스트

색인이 제대로 작동하는지 확인

In [ ]:
from src.retrieval.rag_retriever import RAGRetriever

retriever = RAGRetriever(
    chroma_path=f'{DB_DIR}/chroma_rag',
    use_reranker=False,  # 색인 검증 단계에서는 reranker 생략
)

# 테스트 질의
query = "삼성전자의 사업 부문은 어떻게 구성되는가?"
results = retriever.search(query, k=3)

print(f"🔍 질의: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"=== 결과 {i} (점수: {doc.score:.3f}) ===")
    print(f"출처: {doc.source}")
    print(f"내용: {doc.text[:200]}...\n")

## 3.4 다음 단계

파이프라인 데모를 실행합니다.

→ **04_pipeline_demo.ipynb** 로 진행